# End-to-End Sales Forecasting & Demand Intelligence System
> **Internship Project — Week 3 & Week 4**  
> Dataset: Superstore Sales Dataset (Kaggle) + Video Game Sales (Kaggle)  

---

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import os

# Create output directories
for d in ['../outputs/plots', '../outputs/data', '../outputs/models']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('Imports complete. Environment ready.')

Imports complete. Environment ready.


---
## Task 1 — Data Loading, Merging & Deep Exploration

In [ ]:
# Load the Superstore Sales Dataset
df = pd.read_csv('../data/train.csv', encoding='latin-1', parse_dates=['Order Date', 'Ship Date'], dayfirst=False)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# Load the Video Game Sales Dataset (secondary)
vg = pd.read_csv('../data/vgsales.csv')
print(f'VG Sales Shape: {vg.shape}')
vg.head(3)

In [ ]:
# Data Type Inspection & Quality Check
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicate Rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

In [ ]:
# Feature Engineering — Extract Time Features
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.month
df['Week']       = df['Order Date'].dt.isocalendar().week.astype(int)
df['DayOfWeek']  = df['Order Date'].dt.dayofweek
df['Quarter']    = df['Order Date'].dt.quarter

def get_season(month):
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else:                    return 'Fall'

df['Season'] = df['Month'].apply(get_season)
df['ShipDays'] = (df['Ship Date'] - df['Order Date']).dt.days

print('Time features added.')
df[['Order Date', 'Year', 'Month', 'Week', 'Quarter', 'Season', 'ShipDays']].head(5)

In [ ]:
# Aggregate: Monthly & Weekly Sales

# Monthly aggregation (primary)
monthly_sales = df.groupby(pd.Grouper(key='Order Date', freq='MS'))['Sales'].sum().reset_index()
monthly_sales.columns = ['ds', 'y']
monthly_sales = monthly_sales[monthly_sales['y'] > 0].copy()

# Weekly aggregation
weekly_sales = df.groupby(pd.Grouper(key='Order Date', freq='W'))['Sales'].sum().reset_index()
weekly_sales.columns = ['ds', 'y']
weekly_sales = weekly_sales[weekly_sales['y'] > 0].copy()

print(f'Monthly data points: {len(monthly_sales)}')
print(f'Weekly data points: {len(weekly_sales)}')
monthly_sales.tail(5)

In [ ]:
# Business Question 1: Category Revenue
cat_rev = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
bars = plt.bar(cat_rev.index, cat_rev.values, width=0.5)
plt.bar_label(bars, fmt='$%.0f', padding=5)
plt.title('Total Revenue by Product Category')
plt.ylabel('Total Sales ($)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

print("\nHighest revenue category:")
print(f"{cat_rev.index[0]} — ${cat_rev.iloc[0]:,.0f}")

In [ ]:
# Business Question 2: Most Consistent Regional Growth
region_yearly = df.groupby(['Region', 'Year'])['Sales'].sum().unstack()

plt.figure(figsize=(10, 5))

for region in region_yearly.index:
    plt.plot(region_yearly.columns, region_yearly.loc[region], marker='o', linewidth=2, label=region)

plt.title('Annual Sales by Region (2014–2017)')
plt.ylabel('Total Sales ($)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.legend()
plt.tight_layout()
plt.show()

# Find the most consistent region
growth = region_yearly.pct_change(axis=1)
growth_cv = growth.std(axis=1) / growth.mean(axis=1)
most_consistent = growth_cv.idxmin()

print("\nMost consistent growth region:")
print(most_consistent)

In [ ]:
# Business Question 3: Average Shipping Time by Region
ship_time = df.groupby('Region')['ShipDays'].agg(['mean', 'std']).round(2)
ship_time.columns = ['Avg Ship Days', 'Std Dev']

plt.figure(figsize=(8, 4))
bars = plt.bar(ship_time.index, ship_time['Avg Ship Days'], yerr=ship_time['Std Dev'], capsize=5, width=0.5)
plt.bar_label(bars, fmt='%.1f days', padding=5)
plt.title('Average Shipping Time by Region')
plt.ylabel('Days')
plt.tight_layout()
plt.show()

print("\nShipping time by region:")
print(ship_time)

In [ ]:
# Business Question 4: Seasonal Monthly Spikes
monthly_avg = (df.groupby(['Year', 'Month'])['Sales'].sum().groupby('Month').mean())
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(12, 5))
plt.bar(months, monthly_avg.values)
plt.plot(months, monthly_avg.values, marker='o', linewidth=2)
plt.title('Average Monthly Sales')
plt.ylabel('Average Sales ($)')
plt.gca().yaxis.set_major_formatter( plt.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.show()

top_months = monthly_avg.nlargest(3)
print("\nTop 3 months with the highest average sales:")
for m, v in top_months.items():
    print(f"Month {m:02d} ({months[m-1]}) — ${v:,.0f}")

In [ ]:
# Merging with Video Game Sales (multi-source practice)
# Normalize year columns for join
vg_clean = vg.dropna(subset=['Year']).copy()
vg_clean['Year'] = vg_clean['Year'].astype(int)

superstore_by_year = df.groupby('Year')['Sales'].sum().reset_index()
superstore_by_year.columns = ['Year', 'Superstore_Sales']

vg_by_year = vg_clean.groupby('Year')['Global_Sales'].sum().reset_index()
vg_by_year.columns = ['Year', 'VG_Global_Sales_M']

merged = pd.merge(superstore_by_year, vg_by_year, on='Year', how='inner')

print(' Multi-source merge complete.')
print(f'Merged shape: {merged.shape}')
print('\nNote: Superstore covers 2014-2017. VG data overlaps in those years.')
merged

In [ ]:
# Save processed data for Streamlit
monthly_sales.to_csv('../outputs/data/monthly_sales.csv', index=False)
weekly_sales.to_csv('../outputs/data/weekly_sales.csv', index=False)
df.to_csv('../outputs/data/superstore_processed.csv', index=False)
print(' Processed data saved to outputs/data/')

---
## Task 2 — Time Series Analysis & Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

# Monthly Sales Trend Plot
plt.figure(figsize=(14, 5))
plt.plot(monthly_sales['ds'], monthly_sales['y'], linewidth=2)
plt.fill_between(monthly_sales['ds'], monthly_sales['y'], alpha=0.1)
plt.title('Overall Monthly Sales Trend (2014–2018)')
plt.ylabel('Total Sales ($)')
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Time Series Decomposition
ts_series = monthly_sales.set_index('ds')['y']
ts_series.index = pd.DatetimeIndex(ts_series.index, freq='MS')
decomp = seasonal_decompose(ts_series, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
components = [(ts_series, 'Observed'), (decomp.trend, 'Trend'), (decomp.seasonal, 'Seasonal'), (decomp.resid, 'Residual')]

for ax, (data, label) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=1.8)
    ax.set_ylabel(label)
    ax.grid(True)

axes[0].set_title('Time Series Decomposition - Monthly Sales')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Observations from Decomposition
print("""
Observations from Time Series Decomposition:

1. TREND: The trend component shows a clear upward trajectory from 2014 to 2017,
   indicating steady business growth year-over-year. There is no plateau, suggesting
   the business was still in a growth phase at the end of the observation period.

2. SEASONALITY: Seasonality is moderate-to-strong with consistent peaks around Q4
   (October–December) each year, likely driven by holiday shopping. There is a
   secondary peak around Q3 (September), possibly back-to-school demand.

3. RESIDUAL NOISE: The residual component shows the largest variance in months where
   actual sales deviated sharply from expected (trend + season). The highest residual
   noise appears in Nov/Dec — potentially driven by flash sales or promotions that
   are not fully captured by the seasonal pattern alone.

4. STRENGTH OF SEASONALITY: Seasonal amplitude (~$20K-$40K swing) relative to trend
   (~$200K level) suggests seasonality is real but not dominant — the underlying
   growth trend is the primary driver of revenue.
""")

In [ ]:
# ADF Stationarity Test
def adf_test(series, label='Series'):
    result = adfuller(series.dropna())
    print(f'\n=== ADF Test: {label} ===')
    print(f'ADF Statistic : {result[0]:.4f}')
    print(f'p-value       : {result[1]:.6f}')
    print(f'Critical Values:')
    for k, v in result[4].items():
        print(f'   {k}: {v:.3f}')
    if result[1] < 0.05:
        print('STATIONARY (p < 0.05 → reject null hypothesis of unit root)')
    else:
        print('NON-STATIONARY (p ≥ 0.05 → fail to reject null hypothesis)')
    return result[1]

print("""
What is Stationarity?
A time series is stationary if its statistical properties (mean, variance) do not
change over time. Most forecasting models assume stationarity. If a series has a
trend or growing variance, we must transform it (via differencing) before modeling.
""")

p_orig = adf_test(ts_series, 'Original Monthly Sales')

In [ ]:
# Differencing if Non-Stationary
ts_diff = ts_series.diff().dropna()
p_diff = adf_test(ts_diff, '1st-Order Differenced Sales')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(ts_series.index, ts_series.values, linewidth=1.8)
axes[0].set_title('Original Series')
axes[1].plot(ts_diff.index, ts_diff.values, linewidth=1.8)
axes[1].axhline(0)
axes[1].set_title('After 1st-Order Differencing')
for ax in axes:
    ax.grid(True)
plt.suptitle('Stationarity Check: Before vs After Differencing')
plt.tight_layout()
plt.show()

---
## Task 3 — Sales Forecasting using 3 Different Models

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(actual, predicted):
    actual, predicted = np.array(actual), np.array(predicted)
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

# Train/test split — last 3 months as test
FORECAST_HORIZON = 3
train_data = monthly_sales.iloc[:-FORECAST_HORIZON].copy()
test_data  = monthly_sales.iloc[-FORECAST_HORIZON:].copy()

print(f'Training period: {train_data["ds"].min().date()} → {train_data["ds"].max().date()}')
print(f'Test period:     {test_data["ds"].min().date()} → {test_data["ds"].max().date()}')
print(f'Training size: {len(train_data)} months | Test size: {len(test_data)} months')

results = {}  # Store all model results

In [ ]:
# MODEL 1 — SARIMA (Statistical Model)
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ACF/PACF to select parameters
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ts_diff.dropna(), lags=20, ax=axes[0])
plot_pacf(ts_diff.dropna(), lags=20, ax=axes[1])
axes[0].set_title('ACF - Differenced Sales')
axes[1].set_title('PACF - Differenced Sales')
plt.suptitle('ACF & PACF for SARIMA Parameter Selection')
plt.tight_layout()
plt.show()

In [ ]:
# SARIMA Parameter Rationale:
# p=1 (PACF cuts off after lag 1 in differenced series)
# d=1 (series was non-stationary, 1st-order differencing made it stationary)
# q=1 (ACF cuts off after lag 1)
# P=1, D=1, Q=1 (seasonal component with period m=12 months)
print("""
SARIMA Parameter Selection Rationale:
- p=1 → PACF shows one significant lag after differencing
- d=1 → Series required 1 order of differencing for stationarity (ADF confirmed)
- q=1 → ACF shows one significant lag after differencing
- P=1 → Seasonal AR term (captures year-over-year pattern)
- D=1 → Seasonal differencing (remove annual trend)
- Q=1 → Seasonal MA term
- m=12 → Monthly data with annual seasonality
""")

ts_train = train_data.set_index('ds')['y']
ts_train.index = pd.DatetimeIndex(ts_train.index, freq='MS')

sarima_model = SARIMAX(ts_train,
                       order=(1, 1, 1),
                       seasonal_order=(1, 1, 1, 12),
                       enforce_stationarity=False,
                       enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False)
print('SARIMA model fitted.')
print(sarima_fit.summary())

In [ ]:
# SARIMA Forecast
sarima_forecast = sarima_fit.get_forecast(steps=FORECAST_HORIZON)
sarima_pred = sarima_forecast.predicted_mean
sarima_ci   = sarima_forecast.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts_train.index, ts_train.values, color=PALETTE[0], label='Training', linewidth=2)
ax.plot(test_data['ds'], test_data['y'], color='white', label='Actual Test', linewidth=2, linestyle='--')
ax.plot(sarima_pred.index, sarima_pred.values, color=PALETTE[2], label='SARIMA Forecast', linewidth=2.5, marker='o')
ax.fill_between(sarima_ci.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1],
                alpha=0.2, color=PALETTE[2], label='95% CI')
ax.set_title('SARIMA — Actual vs Forecast (3-Month)', fontsize=14, color='#7C3AED', fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../outputs/plots/sarima_forecast.png', bbox_inches='tight')
plt.show()

# Metrics
test_actual = test_data['y'].values
sarima_vals = sarima_pred.values[:FORECAST_HORIZON]
s_mae  = mean_absolute_error(test_actual, sarima_vals)
s_rmse = np.sqrt(mean_squared_error(test_actual, sarima_vals))
s_mape = mape(test_actual, sarima_vals)
results['SARIMA'] = {'MAE': s_mae, 'RMSE': s_rmse, 'MAPE': s_mape,
                     'F1': sarima_vals[0], 'F2': sarima_vals[1], 'F3': sarima_vals[2]}
print(f'SARIMA → MAE: ${s_mae:,.0f} | RMSE: ${s_rmse:,.0f} | MAPE: {s_mape:.1f}%')

In [ ]:
# MODEL 2 — Facebook Prophet

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    print('Prophet not installed. Run: pip install prophet')
    PROPHET_AVAILABLE = False

In [ ]:
if PROPHET_AVAILABLE:
    prophet_model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,   # monthly data — no weekly component
        daily_seasonality=False,
        seasonality_mode='additive',
        changepoint_prior_scale=0.05  # slightly flexible trend
    )
    prophet_model.fit(train_data)

    # Future dataframe: 3 months ahead
    future = prophet_model.make_future_dataframe(periods=FORECAST_HORIZON, freq='MS')
    prophet_fc = prophet_model.predict(future)

    # Plot
    fig = prophet_model.plot(prophet_fc)
    fig.set_facecolor('#0F0F1A')
    ax = fig.axes[0]
    ax.set_facecolor('#1A1A2E')
    ax.set_title('Prophet — Sales Forecast with Trend & Uncertainty', fontsize=13, color='#7C3AED', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/plots/prophet_forecast.png', bbox_inches='tight')
    plt.show()

    # Seasonality components
    fig2 = prophet_model.plot_components(prophet_fc)
    fig2.set_facecolor('#0F0F1A')
    for ax in fig2.axes:
        ax.set_facecolor('#1A1A2E')
        ax.tick_params(colors='#E2E8F0')
    plt.tight_layout()
    plt.savefig('../outputs/plots/prophet_components.png', bbox_inches='tight')
    plt.show()

    # Metrics on test period
    prophet_test = prophet_fc[prophet_fc['ds'].isin(test_data['ds'])]
    prophet_vals = prophet_test['yhat'].values[:FORECAST_HORIZON]
    # Ensure we have values (fill if needed)
    if len(prophet_vals) < FORECAST_HORIZON:
        future3 = prophet_model.make_future_dataframe(periods=FORECAST_HORIZON + len(monthly_sales), freq='MS')
        prophet_fc2 = prophet_model.predict(future3)
        prophet_vals = prophet_fc2.tail(FORECAST_HORIZON)['yhat'].values

    p_mae  = mean_absolute_error(test_actual, prophet_vals)
    p_rmse = np.sqrt(mean_squared_error(test_actual, prophet_vals))
    p_mape = mape(test_actual, prophet_vals)
    results['Prophet'] = {'MAE': p_mae, 'RMSE': p_rmse, 'MAPE': p_mape,
                          'F1': prophet_vals[0], 'F2': prophet_vals[1], 'F3': prophet_vals[2]}
    print(f'Prophet → MAE: ${p_mae:,.0f} | RMSE: ${p_rmse:,.0f} | MAPE: {p_mape:.1f}%')
else:
    results['Prophet'] = {'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan,
                          'F1': np.nan, 'F2': np.nan, 'F3': np.nan}

In [ ]:
# MODEL 3 — XGBoost for Time Series (ML-based)
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

def create_lag_features(series_df, lags=[1, 2, 3], window=3):
    """Create lag features and rolling stats for supervised ML."""
    data = series_df.copy()
    data['Year']    = data['ds'].dt.year
    data['Month']   = data['ds'].dt.month
    data['Quarter'] = data['ds'].dt.quarter
    data['Season']  = data['Month'].apply(get_season)

    le = LabelEncoder()
    data['Season_enc'] = le.fit_transform(data['Season'])

    for lag in lags:
        data[f'lag_{lag}'] = data['y'].shift(lag)
    data[f'rolling_mean_{window}'] = data['y'].shift(1).rolling(window).mean()
    data = data.dropna()
    return data, le

# Create features on full monthly data
ml_data, le_season = create_lag_features(monthly_sales)

feature_cols = ['Year', 'Month', 'Quarter', 'Season_enc', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_3']
X = ml_data[feature_cols]
y_ml = ml_data['y']

# Train/test split (last 3 months)
X_train, X_test = X.iloc[:-FORECAST_HORIZON], X.iloc[-FORECAST_HORIZON:]
y_train, y_test = y_ml.iloc[:-FORECAST_HORIZON], y_ml.iloc[-FORECAST_HORIZON:]

xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)
print('XGBoost model fitted.')

In [ ]:
# XGBoost Predictions
xgb_pred = xgb_model.predict(X_test)

fig, ax = plt.subplots(figsize=(14, 5))
all_dates = ml_data['ds']
all_actual = ml_data['y']
ax.plot(all_dates[:-FORECAST_HORIZON], all_actual.values[:-FORECAST_HORIZON], label='Training Actual', linewidth=2)
ax.plot(all_dates[-FORECAST_HORIZON:], y_test.values, color='white', label='Test Actual', linewidth=2, linestyle='--')
ax.plot(all_dates[-FORECAST_HORIZON:], xgb_pred, label='XGBoost Forecast', linewidth=2.5, marker='s')
ax.set_title('XGBoost — Actual vs Forecast (3-Month)', fontsize=14, color='#7C3AED', fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../outputs/plots/xgboost_forecast.png', bbox_inches='tight')
plt.show()

# Feature importance
fig, ax = plt.subplots(figsize=(8, 4))
feat_imp = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values()
feat_imp.plot(kind='barh', ax=ax)
ax.set_title('XGBoost Feature Importance', color='#7C3AED', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plots/xgboost_feature_importance.png', bbox_inches='tight')
plt.show()

xgb_vals = xgb_pred[:FORECAST_HORIZON]
x_mae  = mean_absolute_error(test_actual, xgb_vals)
x_rmse = np.sqrt(mean_squared_error(test_actual, xgb_vals))
x_mape = mape(test_actual, xgb_vals)
results['XGBoost'] = {'MAE': x_mae, 'RMSE': x_rmse, 'MAPE': x_mape,
                      'F1': xgb_vals[0], 'F2': xgb_vals[1], 'F3': xgb_vals[2]}
print(f'XGBoost → MAE: ${x_mae:,.0f} | RMSE: ${x_rmse:,.0f} | MAPE: {x_mape:.1f}%')

In [ ]:
# 3.4 Model Comparison Table
forecast_dates = test_data['ds'].dt.strftime('%b %Y').tolist()

comparison_df = pd.DataFrame({
    'Model'          : list(results.keys()),
    'MAE ($)'        : [f"${v['MAE']:,.0f}" for v in results.values()],
    'RMSE ($)'       : [f"${v['RMSE']:,.0f}" for v in results.values()],
    'MAPE (%)'       : [f"{v['MAPE']:.1f}%" for v in results.values()],
    f'Forecast {forecast_dates[0]}' : [f"${v['F1']:,.0f}" for v in results.values()],
    f'Forecast {forecast_dates[1]}' : [f"${v['F2']:,.0f}" for v in results.values()],
    f'Forecast {forecast_dates[2]}' : [f"${v['F3']:,.0f}" for v in results.values()],
}).set_index('Model')

print('\n' + '='*80)
print('                    MODEL COMPARISON TABLE')
print('='*80)
print(comparison_df.to_string())
print('='*80)

# Best model by MAPE
best_model = min(results, key=lambda k: results[k]['MAPE'] if not np.isnan(results[k]['MAPE']) else float('inf'))
print(f'\n🏆 Best Model: {best_model} (lowest MAPE: {results[best_model]["MAPE"]:.1f}%)')

# Save comparison for Streamlit
comparison_df.to_csv('../outputs/data/model_comparison.csv')
import json
with open('../outputs/data/model_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"""
Production Recommendation:
The {best_model} model is recommended for production because it achieves the lowest
forecasting error on held-out test data.

- SARIMA: Excellent for well-behaved seasonal data but struggles with sudden regime
  changes (e.g., promotions, stockouts). Interpretable, but manual parameter tuning.

- Prophet: Robust to missing data and outliers, automatically handles holidays. Best
  choice when the business calendar matters (festive sales, campaigns).

- XGBoost: Most flexible — can incorporate external regressors (weather, marketing
  spend). Requires more feature engineering but scales to product-level forecasting.

Recommendation: Use the best-performing model as the primary forecast engine,
with Prophet as a human-interpretable backup for stakeholder presentations.
""")

---
## Task 4 — Product Category & Region Level Forecasting

In [ ]:
import joblib

segments = {
    'Furniture':       df[df['Category']=='Furniture'],
    'Technology':      df[df['Category']=='Technology'],
    'Office Supplies': df[df['Category']=='Office Supplies'],
    'West Region':     df[df['Region']=='West'],
    'East Region':     df[df['Region']=='East'],
}

segment_forecasts = {}

for seg_name, seg_df in segments.items():
    seg_monthly = seg_df.groupby(pd.Grouper(key='Order Date', freq='MS'))['Sales'].sum().reset_index()
    seg_monthly.columns = ['ds', 'y']
    seg_monthly = seg_monthly[seg_monthly['y'] > 0].reset_index(drop=True)

    seg_ml, _ = create_lag_features(seg_monthly)
    X_seg = seg_ml[feature_cols]
    y_seg = seg_ml['y']

    X_tr = X_seg.iloc[:-FORECAST_HORIZON]
    y_tr = y_seg.iloc[:-FORECAST_HORIZON]
    X_te = X_seg.iloc[-FORECAST_HORIZON:]

    seg_xgb = xgb.XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.05,
                                random_state=42, verbosity=0)
    seg_xgb.fit(X_tr, y_tr)
    seg_pred = seg_xgb.predict(X_te)

    segment_forecasts[seg_name] = {
        'dates': seg_ml['ds'].tail(FORECAST_HORIZON).tolist(),
        'actual': y_seg.tail(FORECAST_HORIZON).tolist(),
        'pred':   seg_pred.tolist(),
        'historical_dates': seg_ml['ds'].tolist(),
        'historical_y':     y_seg.tolist(),
    }
    joblib.dump(seg_xgb, f'../outputs/models/xgb_{seg_name.replace(" ", "_")}.pkl')
    print(f'{seg_name}: forecast complete')

In [ ]:
# Combined comparison chart
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, (seg_name, seg_data) in enumerate(segment_forecasts.items()):
    ax = axes[idx]
    hist_d = [pd.Timestamp(d) for d in seg_data['historical_dates']]
    fore_d = [pd.Timestamp(d) for d in seg_data['dates']]

    ax.plot(hist_d[:-FORECAST_HORIZON], seg_data['historical_y'][:-FORECAST_HORIZON], linewidth=2, label='Historical')
    ax.plot(fore_d, seg_data['actual'], color='white', linewidth=2, linestyle='--', label='Actual')
    ax.plot(fore_d, seg_data['pred'], linewidth=2.5, marker='o', label='Forecast')
    ax.set_title(seg_name, color='#E2E8F0', fontsize=11, fontweight='bold')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_visible(False)  # hide 6th subplot (only 5 segments)
fig.suptitle('Segment-Level Forecasts — Categories & Regions', fontsize=16,
             color='#7C3AED', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/plots/segment_forecasts.png', bbox_inches='tight')
plt.show()

# Growth analysis
print('\n Segment Growth Analysis (Forecast vs Last Actual):')
for seg_name, seg_data in segment_forecasts.items():
    last_actual = seg_data['historical_y'][-FORECAST_HORIZON-1]
    avg_forecast = np.mean(seg_data['pred'])
    growth = (avg_forecast - last_actual) / last_actual * 100
    print(f'   {seg_name:<20} → Avg Forecast: ${avg_forecast:,.0f} | Growth: {growth:+.1f}%')

import json
# Serialize for Streamlit
seg_for_json = {k: {kk: ([str(x) for x in vv] if isinstance(vv, list) and len(vv)>0 and hasattr(vv[0], 'strftime') else vv)
                    for kk, vv in v.items()} for k, v in segment_forecasts.items()}
with open('../outputs/data/segment_forecasts.json', 'w') as f:
    json.dump(seg_for_json, f, indent=2, default=str)

---
## Task 5 — Anomaly Detection in Sales Data

In [ ]:
from sklearn.ensemble import IsolationForest

# Prepare Weekly Sales for Anomaly Detection
weekly = weekly_sales.copy()
weekly = weekly[weekly['y'] > 0].reset_index(drop=True)

# Isolation Forest
iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,  # expect ~5% anomalies
    random_state=42
)
weekly['iso_score'] = iso_model.fit_predict(weekly[['y']])
weekly['iso_anomaly'] = weekly['iso_score'] == -1

print(f'Isolation Forest detected {weekly["iso_anomaly"].sum()} anomaly weeks out of {len(weekly)}')

In [2]:
# Z-Score Anomaly Detection
rolling_mean = weekly['y'].rolling(window=8, min_periods=4, center=True).mean()
rolling_std  = weekly['y'].rolling(window=8, min_periods=4, center=True).std()
weekly['z_score'] = (weekly['y'] - rolling_mean) / rolling_std
weekly['z_anomaly'] = weekly['z_score'].abs() > 2.0

print(f'Z-Score detected {weekly["z_anomaly"].sum()} anomaly weeks out of {len(weekly)}')

NameError: name 'weekly' is not defined

In [ ]:
# Anomaly Plot
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Plot 1: Isolation Forest
ax1 = axes[0]
ax1.plot(weekly['ds'], weekly['y'], linewidth=1.5, label='Weekly Sales')
ax1.scatter(weekly.loc[weekly['iso_anomaly'], 'ds'], weekly.loc[weekly['iso_anomaly'], 'y'], s=80, zorder=5, label='Anomaly (Isolation Forest)', edgecolors='white', linewidth=0.5)
ax1.set_title('Anomaly Detection — Isolation Forest', fontsize=13, color='#7C3AED', fontweight='bold')
ax1.legend()
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax1.grid(True, alpha=0.3)

# Plot 2: Z-Score
ax2 = axes[1]
ax2.plot(weekly['ds'], weekly['y'], color=PALETTE[1], linewidth=1.5, label='Weekly Sales')
ax2.fill_between(weekly['ds'], rolling_mean - 2*rolling_std, rolling_mean + 2*rolling_std, alpha=0.2, label='±2σ Band')
ax2.plot(weekly['ds'], rolling_mean, color='white', linewidth=1, linestyle='--', alpha=0.6, label='Rolling Mean')
ax2.scatter(weekly.loc[weekly['z_anomaly'], 'ds'], weekly.loc[weekly['z_anomaly'], 'y'], s=80, zorder=5, label='Anomaly (Z-Score)', edgecolors='white', linewidth=0.5)
ax2.set_title('Anomaly Detection — Z-Score (Rolling ±2σ)', fontsize=13, color='#7C3AED', fontweight='bold')
ax2.legend()
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/plots/anomaly_detection.png', bbox_inches='tight')
plt.show()

In [ ]:
# Anomaly Explanation Table
anomaly_weeks = weekly[weekly['iso_anomaly'] | weekly['z_anomaly']].copy()
anomaly_weeks['Month_Name'] = pd.to_datetime(anomaly_weeks['ds']).dt.strftime('%b %Y')
anomaly_weeks['Detected By'] = anomaly_weeks.apply(
    lambda r: 'Both' if r['iso_anomaly'] and r['z_anomaly']
              else ('Isolation Forest' if r['iso_anomaly'] else 'Z-Score'), axis=1)

# Auto-generate explanations based on month
def explain_anomaly(row):
    month = pd.to_datetime(row['ds']).month
    sales = row['y']
    avg   = weekly['y'].mean()
    if sales > avg * 1.5:
        if month in [11, 12]: return 'Holiday/festive season — Black Friday, Christmas promotions'
        elif month in [8, 9]: return 'Back-to-school demand spike in Office Supplies'
        elif month in [3, 4]: return 'Spring promotion or quarterly corporate purchasing'
        else:                 return 'Unusually high demand — possible bulk order or campaign'
    else:
        if month in [1, 2]:   return 'Post-holiday lull — lowest demand period'
        elif month in [6, 7]: return 'Summer slowdown — reduced corporate purchasing'
        else:                 return 'Unusually low demand — possible supply issue or data gap'

anomaly_weeks['Likely Cause'] = anomaly_weeks.apply(explain_anomaly, axis=1)

display_cols = ['ds', 'y', 'Detected By', 'Likely Cause']
print(anomaly_weeks[display_cols].to_string(index=False))

anomaly_weeks[display_cols].to_csv('../outputs/data/anomalies.csv', index=False)

print(f"""
Method Comparison:
- Isolation Forest is density-based and global — it flags points that are statistically
  isolated from the bulk of the data. Good at catching absolute outliers.
- Z-Score is context-aware via the rolling window — it flags points that deviate from
  their local neighborhood. Better at detecting seasonal anomalies.
- When both agree: high confidence anomaly (likely a real event)
- When they disagree: borderline case — may need domain knowledge to confirm
""")

---
## Task 6 — Product Demand Segmentation using Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Build Sub-category Feature Matrix
df['Year_Month'] = df['Order Date'].dt.to_period('M')

subcat_monthly = df.groupby(['Sub-Category', 'Year_Month'])['Sales'].sum().reset_index()

# Features per sub-category
subcat_features = df.groupby('Sub-Category').agg(
    Total_Sales   = ('Sales', 'sum'),
    Avg_Order_Val = ('Sales', 'mean'),
    Sales_Std     = ('Sales', 'std'),
    Order_Count   = ('Order ID', 'nunique'),
).reset_index()

# Year-over-year growth rate
yoy = df.groupby(['Sub-Category', 'Year'])['Sales'].sum().unstack(fill_value=0)
if 2016 in yoy.columns and 2017 in yoy.columns:
    yoy['Growth_Rate'] = (yoy[2017] - yoy[2016]) / yoy[2016].replace(0, np.nan) * 100
else:
    years = sorted(yoy.columns)
    yoy['Growth_Rate'] = (yoy[years[-1]] - yoy[years[-2]]) / yoy[years[-2]].replace(0, np.nan) * 100

subcat_features = subcat_features.merge(yoy[['Growth_Rate']].reset_index(), on='Sub-Category', how='left')
subcat_features['Growth_Rate'] = subcat_features['Growth_Rate'].fillna(0)

print('Sub-category features built:')
subcat_features.head(10)

In [ ]:
# Elbow Method
feature_cols_cluster = ['Total_Sales', 'Avg_Order_Val', 'Sales_Std', 'Order_Count', 'Growth_Rate']
X_cluster = subcat_features[feature_cols_cluster].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_range), inertias, marker='o', linewidth=2.5, markersize=8)
ax.set_title('Elbow Method — Optimal K for K-Means', fontsize=13, color='#7C3AED', fontweight='bold')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia (Within-Cluster SSE)')
ax.axvline(x=4, linestyle='--', alpha=0.7, label='Selected K=4')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/plots/elbow_method.png', bbox_inches='tight')
plt.show()
print('Selected K=4 based on the elbow point (diminishing returns after 4 clusters).')

In [ ]:
# K-Means Clustering
K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
subcat_features['Cluster'] = kmeans.fit_predict(X_scaled)

# PCA for 2D Visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
subcat_features['PC1'] = X_pca[:, 0]
subcat_features['PC2'] = X_pca[:, 1]

# Label clusters based on centroid characteristics
cluster_centroids = subcat_features.groupby('Cluster')[feature_cols_cluster].mean()
print('\nCluster Centroids (scaled back to original space):')
print(cluster_centroids.round(0))

In [ ]:
# Assign Meaningful Labels
# Rank clusters by Total_Sales to assign labels
cluster_rank = cluster_centroids['Total_Sales'].rank(ascending=False).astype(int)
growth_rank  = cluster_centroids['Growth_Rate'].rank(ascending=False).astype(int)
vol_rank     = cluster_centroids['Sales_Std'].rank(ascending=False).astype(int)

def label_cluster(c):
    ts = cluster_centroids.loc[c, 'Total_Sales']
    gr = cluster_centroids.loc[c, 'Growth_Rate']
    std = cluster_centroids.loc[c, 'Sales_Std']
    if gr > 20:       return 'Growing Demand'
    elif gr < -10:    return 'Declining Demand'
    elif std > cluster_centroids['Sales_Std'].mean():
                      return 'High Volatility'
    elif ts > cluster_centroids['Total_Sales'].mean():
                      return 'High Volume, Stable'
    else:             return 'Low Volume, Stable'

cluster_labels = {c: label_cluster(c) for c in range(K)}
subcat_features['Cluster_Label'] = subcat_features['Cluster'].map(cluster_labels)

print('\nCluster Label Mapping:')
for c, label in cluster_labels.items():
    members = subcat_features[subcat_features['Cluster']==c]['Sub-Category'].tolist()
    print(f'  Cluster {c} — {label}: {members}')

In [ ]:
# PCA Scatter Plot
fig, ax = plt.subplots(figsize=(11, 7))
for label in subcat_features['Cluster_Label'].unique():
    mask = subcat_features['Cluster_Label'] == label
    ax.scatter(subcat_features.loc[mask, 'PC1'], subcat_features.loc[mask, 'PC2'], s=120, label=label, alpha=0.9)

for _, row in subcat_features.iterrows():
    ax.annotate(row['Sub-Category'], (row['PC1'] + 0.05, row['PC2'] + 0.05), fontsize=7)

ax.set_title('Product Demand Segmentation (K-Means + PCA)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)')
ax.legend(title='Demand Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# Stocking Strategy per Cluster
print("""
STOCKING STRATEGY PER DEMAND CLUSTER
═══════════════════════════════════════════

1. HIGH VOLUME, STABLE DEMAND
   Strategy: Maintain high safety stock. Use automated reorder points (ROP).
   These products drive the most revenue — stockouts are very costly.
   Action: Use Just-in-Time restocking with minimum buffer of 4–6 weeks supply.

2. GROWING DEMAND
   Strategy: Proactively increase stock levels quarter-over-quarter.
   Monitor sell-through rate monthly. Negotiate better supplier terms now
   before demand peaks and prices rise.
   Action: Increase order quantities by 15–20% per quarter; add secondary supplier.

3. HIGH VOLATILITY
   Strategy: Keep higher safety stock (1.5–2x normal) to absorb demand spikes.
   Avoid overcommitting to long-term purchase orders.
   Action: Use flexible, shorter-term supplier contracts; review stock weekly.

4. DECLINING DEMAND
   Strategy: Reduce stock levels. Run clearance promotions to move inventory.
   Do not reorder until current stock drops below 2 weeks supply.
   Action: Flag for product rationalization — consider discontinuing underperformers.
""")

# Save for Streamlit
subcat_features.to_csv('../outputs/data/demand_segments.csv', index=False)
print('Demand segmentation saved.')

In [ ]:
# Save Global XGBoost model
joblib.dump(xgb_model, '../outputs/models/xgb_global.pkl')
joblib.dump(le_season, '../outputs/models/label_encoder.pkl')
joblib.dump(kmeans,    '../outputs/models/kmeans.pkl')
joblib.dump(scaler,    '../outputs/models/scaler.pkl')
joblib.dump(pca,       '../outputs/models/pca.pkl')
print('All models saved to outputs/models/')